# AkasicDB 하이브리드 검색: 그래프 + 전문 검색 + 벡터

이 노트북에서는 합성 상품 데이터를 AkasicDB에 적재하고, **그래프(Cypher)**, **전문 검색(BM25, texton)**, **벡터 유사도(vectoron)**, **SQL 필터**를 하나의 SQL 쿼리로 함께 실행합니다. 마지막으로 PostgreSQL의 `EXPLAIN`으로 실행 계획을 확인합니다.

상품 설명은 texton의 한국어 형태소 분석기(lindera + ko-dic)로 색인합니다. "러닝화를", "방수가"처럼 조사가 붙은 표현이 "러닝화", "방수"로 검색되는 것을 Step 9에서 확인합니다.

## 준비물

* AkasicDB 인스턴스
    * [db.akasic.cloud](https://db.akasic.cloud)에서 Playground 인스턴스를 만들거나,
    * [설치 문서](https://docs.graphai.io/akasicdb-docs/latest/installation/installation/)를 따라 로컬에 설치합니다.
* 연결 문자열
    * Google Colab: 왼쪽 사이드바의 열쇠 아이콘을 눌러 `DB_DSN`이라는 이름의 Secret으로 저장합니다.
    * 그 밖의 환경: 환경 변수 `DATABASE_URL`로 지정합니다.
* (선택) GPU 런타임. Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 고릅니다. GPU가 없으면 데이터 규모를 줄여 CPU로 실행합니다.

## Step 1: 패키지 설치

In [ ]:
%pip install --quiet akasicdb "psycopg[binary]" certifi "sentence-transformers>=2.7" "transformers>=4.51" accelerate

## Step 2: DB 연결과 확장 활성화

환경 변수 `DATABASE_URL`이 있으면 그 값을, 없으면 Colab Secrets의 `DB_DSN`을 사용합니다.

`akasicdb` 확장은 데이터베이스마다 한 번 활성화해야 합니다. `CASCADE`는 의존 확장(`vectoron`, `graphon`)을 함께 설치합니다. 이미 활성화된 데이터베이스에서 다시 실행해도 안전합니다.

Playground 연결 문자열은 서버 인증서를 검증하는 `sslmode=verify-full&sslrootcert=system`을 사용합니다. `psycopg[binary]`에 포함된 OpenSSL은 운영체제의 인증서 저장소를 찾지 못할 수 있으므로, `certifi`의 인증서 묶음을 `SSL_CERT_FILE`로 알려 줍니다.

`connect()`는 연결과 세션 설정(벡터 타입 등록, `search_path`)을 함께 처리합니다. 이후 단계에서 연결이 끊어지면 `conn = connect()`를 실행한 뒤 해당 셀을 다시 실행하세요.

확장 생성과 `akasicdb_admin.initialize()`는 관리자 권한이 필요합니다. Playground처럼 관리자가 이미 초기화한 데이터베이스에서 권한이 없으면 초기화를 건너뛰고 진행합니다.

In [ ]:
import os

import certifi
import psycopg
from akasicdb.psycopg import register_vector

# psycopg[binary]에 포함된 OpenSSL은 운영체제의 인증서 저장소를 찾지 못할 수 있습니다.
# sslmode=verify-full&sslrootcert=system 연결(Playground 기본값)이 서버 인증서를 검증할 수 있도록
# certifi의 인증서 묶음 위치를 알려 줍니다. 이미 SSL_CERT_FILE을 설정했다면 그 값을 그대로 씁니다.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())


MSG_NEED_ADMIN = "akasicdb 확장을 만들 권한이 없습니다. 데이터베이스 관리자에게 CREATE EXTENSION akasicdb CASCADE 실행을 요청해 주세요."
MSG_SKIP_INIT = "akasicdb_admin.initialize()를 실행할 권한이 없어 건너뜁니다. 관리자가 이미 초기화한 데이터베이스라면 그대로 진행하면 됩니다."


def get_dsn() -> str:
    dsn = os.getenv("DATABASE_URL")
    if dsn:
        return dsn
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError("환경 변수 DATABASE_URL을 설정하거나 Colab Secrets에 DB_DSN을 저장해 주세요.")
    return userdata.get("DB_DSN")


def configure(conn: psycopg.Connection) -> None:
    """세션 설정: 벡터 타입을 등록하고, 예제 스키마를 search_path 맨 앞에 추가합니다."""
    register_vector(conn)
    conn.execute("SELECT set_config('search_path', 'hybrid_search_demo, ' || current_setting('search_path'), false)")


def connect() -> psycopg.Connection:
    """새로 연결하고 세션 설정을 적용합니다. 연결이 끊어졌을 때 다시 연결하는 데에도 씁니다."""
    conn = psycopg.connect(get_dsn(), autocommit=True)
    configure(conn)
    return conn


conn = psycopg.connect(get_dsn(), autocommit=True)
try:
    conn.execute("CREATE EXTENSION IF NOT EXISTS akasicdb CASCADE")
except psycopg.errors.InsufficientPrivilege:
    raise RuntimeError(MSG_NEED_ADMIN) from None
try:
    conn.execute("SELECT akasicdb_admin.initialize()")
except psycopg.errors.InsufficientPrivilege:
    print(MSG_SKIP_INIT)
configure(conn)

print(conn.execute("SELECT version()").fetchone()[0])

## Step 3: 스키마 생성

예제의 테이블은 전용 스키마 `hybrid_search_demo`에, 그래프는 `hybrid_search_demo_graph`라는 이름으로 만듭니다. 기존 데이터와 섞이지 않도록 하기 위해서입니다. 노트북을 다시 실행할 수 있도록 이 스키마와 그래프는 매번 지우고 새로 만듭니다. 상품 설명 임베딩은 Qwen3-Embedding-0.6B의 출력 차원에 맞춰 `vectoron.vector(1024)`로 저장합니다.

In [ ]:
for stmt in (
    "SELECT akasicdb.delete_graph('hybrid_search_demo_graph', true)",
    "SELECT akasicdb.undefine_graph('hybrid_search_demo_graph')",
):
    try:
        conn.execute(stmt)
    except psycopg.Error:
        pass  # 그래프가 없으면 건너뜁니다.

conn.execute("DROP SCHEMA IF EXISTS hybrid_search_demo CASCADE")
conn.execute("CREATE SCHEMA hybrid_search_demo")

conn.execute("""
CREATE TABLE brands (
  b_id  integer PRIMARY KEY,
  name  text    NOT NULL,
  hot   boolean NOT NULL DEFAULT false
)""")

conn.execute("""
CREATE TABLE products (
  id       integer PRIMARY KEY,
  b_id     integer NOT NULL REFERENCES brands(b_id),
  title    text    NOT NULL,
  category text    NOT NULL,
  price    integer NOT NULL,
  descr    text    NOT NULL,
  emb      vectoron.vector(1024)
)""")
print("ok")

## Step 4: 합성 데이터 생성

GPU가 있으면 상품 10,000개, 없으면 2,000개를 만듭니다. 브랜드 50개 중 10개는 인기(`hot`) 브랜드이고, 인기 브랜드 상품은 모든 카테고리에 고르게 들어갑니다.

상품 설명은 계절, 품목, 특징을 조합한 문장입니다. "러닝화를", "트레킹화는", "방수가"처럼 조사가 붙은 형태가 섞여 있습니다.

In [ ]:
import random

import torch

if torch.cuda.is_available():
    DEVICE, DTYPE, N_PRODUCTS = "cuda", torch.float16, 10_000
else:
    DEVICE, DTYPE, N_PRODUCTS = "cpu", torch.float32, 2_000
print(f"장치: {DEVICE}, 상품 수: {N_PRODUCTS:,}")

random.seed(42)
N_BRANDS = 50

seasons = ["여름", "겨울", "사계절", "봄가을"]
items = ["러닝화", "운동화", "샌들", "트레킹화", "슬립온"]
feats = ["통기성이 좋은", "가볍고 시원한", "쿠션이 좋은", "방수가 되는"]
cats = ["shoes", "apparel", "bag", "acc", "outdoor"]


def josa(word: str, with_batchim: str, without_batchim: str) -> str:
    """받침 유무에 맞는 조사를 붙입니다. 예: 러닝화를, 샌들을"""
    has_batchim = (ord(word[-1]) - 0xAC00) % 28 != 0
    return word + (with_batchim if has_batchim else without_batchim)


templates = [
    lambda s, i, f: f"{s} 시즌 {josa(i, '을', '를')} 찾는 분께 추천하는 {f} 제품입니다.",
    lambda s, i, f: f"{josa(i, '은', '는')} {f} 소재라 {s}에 신기 좋습니다.",
    lambda s, i, f: f"{f} {josa(i, '이', '가')} {s} 내내 편안합니다.",
]

brands = [(b, f"brand_{b}", b % 5 == 0) for b in range(1, N_BRANDS + 1)]

products = []
for g in range(1, N_PRODUCTS + 1):
    season = seasons[(g // 7) % len(seasons)]
    item = items[(g // 3) % len(items)]
    feat = feats[g % len(feats)]
    descr = templates[(g // 11) % len(templates)](season, item, feat)
    products.append((
        g,                                   # id
        (g // 5) % N_BRANDS + 1,             # b_id: 카테고리와 독립적으로 배정
        f"product_{g}",                      # title
        cats[g % len(cats)],                 # category
        10_000 + random.randrange(190_000),  # price
        descr,
    ))

hot_brands = {b for b, _, hot in brands if hot}
hot_shoes = sum(1 for p in products if p[1] in hot_brands and p[3] == "shoes")
print(f"인기 브랜드의 shoes 상품: {hot_shoes}개")
for p in products[:3]:
    print(p)

## Step 5: 임베딩 생성

Qwen3-Embedding-0.6B는 다국어 임베딩 모델입니다. 상품 설명은 instruction 없이 인코딩하고, 검색 질의는 `prompt_name="query"`로 인코딩합니다. `normalize_embeddings=True`로 정규화하면 L2 거리(`<->`) 순서가 코사인 유사도 순서와 같아집니다.

T4 GPU에서 10,000건은 수 분, CPU에서 2,000건은 수 분에서 십여 분 걸립니다.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device=DEVICE,
    model_kwargs={"torch_dtype": DTYPE},
    tokenizer_kwargs={"padding_side": "left"},
)

embs = model.encode(
    [p[5] for p in products],
    batch_size=64 if DEVICE == "cuda" else 16,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(embs.shape)

## Step 6: 데이터 적재

1,000건씩 나눠 적재합니다. 배치마다 하나의 트랜잭션으로 넣고, 적재 중 연결이 끊어지면 `connect()`로 다시 연결해 같은 배치부터 다시 넣습니다(최대 3번). 다시 연결한 세션에도 `search_path`가 설정되므로 이후 단계는 그대로 진행됩니다. `ON CONFLICT DO NOTHING`으로 같은 배치를 두 번 넣어도 중복되지 않습니다.

In [ ]:
import time

MAX_ATTEMPTS = 3


def insert_rows(sql: str, rows: list) -> None:
    """배치를 한 트랜잭션으로 넣습니다. 연결이 끊어지면 다시 연결해 같은 배치를 다시 넣습니다."""
    global conn
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            with conn.transaction(), conn.cursor() as cur:
                cur.executemany(sql, rows)
            return
        except psycopg.OperationalError as e:
            if attempt == MAX_ATTEMPTS:
                raise
            reason = str(e).strip().splitlines()[0]
            print(f"연결이 끊어져 다시 연결합니다 ({attempt}/{MAX_ATTEMPTS}): {reason}")
            time.sleep(2 * attempt)
            conn = connect()


insert_rows("INSERT INTO brands VALUES (%s, %s, %s) ON CONFLICT (b_id) DO NOTHING", brands)

rows = [(*p, emb) for p, emb in zip(products, embs)]
BATCH = 1000
for i in range(0, len(rows), BATCH):
    insert_rows(
        "INSERT INTO products (id, b_id, title, category, price, descr, emb) "
        "VALUES (%s, %s, %s, %s, %s, %s, %s) ON CONFLICT (id) DO NOTHING",
        rows[i:i + BATCH],
    )
    print(f"{min(i + BATCH, len(rows)):,} / {len(rows):,}")

conn.execute("ANALYZE brands, products")
loaded = conn.execute("SELECT count(*) FROM products").fetchone()[0]
print(f"적재된 상품: {loaded:,}개")

## Step 7: 인덱스 생성

세 종류의 인덱스를 만듭니다.

* **vectoron**: 임베딩 벡터의 근사 최근접 검색(ANN)
* **texton**: 상품 설명의 전문 검색(BM25). 행 전체(`products.*`)를 대상으로 하는 whole-row 인덱스이며, `mapping`에서 `descr` 필드를 한국어 형태소 분석기(lindera + ko-dic)로 색인하도록 지정합니다. `@@` 검색 연산자는 이 인덱스를 사용합니다.
* **B-tree**: 카테고리 필터

In [ ]:
conn.execute("CREATE INDEX products_emb_idx ON products USING vectoron (emb vector_l2_ops)")

conn.execute("""
CREATE INDEX products_texton_idx
  ON products
  USING texton ((products.*))
  WITH (mapping='{"fields":{
    "descr":{"type":"text","analyzer":{"tokenizer":{"lindera":{"dictionary":"ko-dic"}}}}
  }}')""")

conn.execute("CREATE INDEX products_cat_idx ON products (category)")
print("ok")

## Step 8: 그래프 정의

기존 `products`, `brands` 테이블에서 그래프의 정점과 간선을 정의합니다. 데이터를 다시 적재하지 않습니다.

* 정점: `v_product`(상품), `v_brand`(브랜드)
* 간선: `e_made_by`(상품 → 브랜드)

커뮤니티 에디션은 데이터베이스마다 그래프를 1개까지 만들 수 있습니다. 이미 다른 그래프가 있으면 기존 그래프를 지우는 방법을 안내하고 멈춥니다.

In [ ]:
MSG_GRAPH_LIMIT = (
    "커뮤니티 에디션은 그래프를 1개까지 만들 수 있는데, 이 데이터베이스에 이미 다른 그래프가 있습니다. "
    "SELECT * FROM akasicdb.list_graph()로 그래프를 확인하고, 필요 없다면 "
    "SELECT akasicdb.delete_graph('<그래프 이름>', true)로 지운 뒤 다시 실행해 주세요."
)

try:
    conn.execute("SELECT akasicdb.define_graph('hybrid_search_demo_graph')")
except psycopg.Error as e:
    if "community edition allows" in str(e):
        raise RuntimeError(MSG_GRAPH_LIMIT) from None
    raise

conn.execute("""
SELECT akasicdb.define_vertex('hybrid_search_demo_graph', 'v_product',
  ARRAY['product_id integer', 'title text'],
  'hybrid_search_demo.products',
  'SELECT id AS product_id, title FROM hybrid_search_demo.products')""")

conn.execute("""
SELECT akasicdb.define_vertex('hybrid_search_demo_graph', 'v_brand',
  ARRAY['b_id integer', 'name text', 'hot boolean'],
  'hybrid_search_demo.brands',
  'SELECT b_id, name, hot FROM hybrid_search_demo.brands')""")

conn.execute("""
SELECT akasicdb.define_edge('hybrid_search_demo_graph', 'e_made_by',
  'v_product', 'v_brand',
  null,
  'SELECT null FROM hybrid_search_demo.products p, hybrid_search_demo.brands b WHERE p.b_id = b.b_id',
  'hybrid_search_demo.products p', 'hybrid_search_demo.brands b')""")

conn.execute("SELECT akasicdb.create_graph('hybrid_search_demo_graph')")
print("graph ready")

## Step 9: 한국어 형태소 분석 효과 확인

비교를 위해 같은 상품 설명을 언어 중립 `standard` analyzer로 색인한 테이블 `products_std`를 만들고, 같은 검색어의 결과 건수를 비교합니다.

`실제`는 설명에 검색어가 문자 그대로 들어 있는 상품 수입니다. `standard`는 공백과 문장부호로만 단어를 나누므로 "러닝화를"을 한 단어로 색인하고, "러닝화"로는 찾지 못합니다. ko-dic은 형태소 단위로 나누므로 조사가 붙은 표현도 찾습니다.

In [ ]:
conn.execute("CREATE TABLE products_std AS SELECT id, descr FROM products")
conn.execute("""
CREATE INDEX products_std_texton_idx
  ON products_std
  USING texton ((products_std.*))
  WITH (mapping='{"fields":{"descr":{"type":"text"}}}')""")

print(f"{'검색어':<8}{'실제':>6}{'ko-dic':>8}{'standard':>10}")
for word in ["러닝화", "방수", "트레킹화"]:
    total = conn.execute(
        "SELECT count(*) FROM products WHERE descr LIKE %s", (f"%{word}%",)
    ).fetchone()[0]
    ko = conn.execute(
        "SELECT count(*) FROM products p WHERE p @@ akasicdb.text_query(%s)", (word,)
    ).fetchone()[0]
    std = conn.execute(
        "SELECT count(*) FROM products_std s WHERE s @@ akasicdb.text_query(%s)", (word,)
    ).fetchone()[0]
    print(f"{word:<8}{total:>6}{ko:>8}{std:>10}")

ko-dic은 조사가 붙은 표현 대부분을 찾지만, 사전 기반 형태소 분석이라 모든 경우를 찾지는 못합니다. 이 데이터에서는 "트레킹화가"가 사전에 따라 다른 형태소로 나뉘어 "트레킹화"로 검색되지 않습니다. 도메인 용어나 신조어는 [사용자 사전](https://docs.graphai.io/akasicdb-docs/latest/usage/text-search-features/tokenization/)으로 색인 단위를 지정할 수 있습니다.

## Step 10: 하이브리드 질의

하나의 SQL 쿼리에서 네 가지 조건을 함께 사용합니다.

1. **그래프(Cypher)**: 인기 브랜드(`hot = true`)가 만든 상품
2. **SQL 필터**: `category = 'shoes'`
3. **전문 검색(BM25)**: 설명에 "여름"과 "러닝화"가 모두 들어간 상품. `akasicdb.text_score(p)`로 BM25 점수도 함께 봅니다.
4. **벡터 유사도**: "여름에 신을 시원한 러닝화"와 의미가 가까운 순서로 정렬. 결과의 `dist`가 질의 벡터와의 L2 거리입니다.

합성 데이터라 BM25 점수가 같은 상품이 많습니다. 이때는 벡터 거리(`dist`)가 순서를 정합니다.

In [ ]:
q_emb = model.encode(
    ["여름에 신을 시원한 러닝화"],
    prompt_name="query",
    normalize_embeddings=True,
)[0]

HYBRID_SQL = """
SELECT p.id, p.title, p.price, akasicdb.text_score(p) AS bm25,
       p.emb <-> %s::vectoron.vector AS dist, left(p.descr, 40) AS descr
FROM   products p
JOIN   akasicdb.cypher('hybrid_search_demo_graph', $$
           MATCH (pr:v_product)-[:e_made_by]->(b:v_brand)
           WHERE b.hot = true
           RETURN pr.product_id AS product_id
       $$) AS g (product_id integer)
       ON g.product_id = p.id                        -- 그래프 (Cypher)
WHERE  p.category = 'shoes'                          -- SQL 필터
  AND  p @@ akasicdb.text_query('여름 AND 러닝화')     -- 전문 검색 (BM25)
ORDER  BY dist                                        -- 벡터 유사도 (L2)
LIMIT  10
"""

rows = conn.execute(HYBRID_SQL, (q_emb,)).fetchall()
for id_, title, price, bm25, dist, descr in rows:
    print(f"{id_:>5}  {title:<13} price={price:>7,}  bm25={bm25:.3f}  dist={dist:.4f}  {descr}")
print(f"하이브리드 결과: {len(rows)}건")

## Step 11: 실행 계획 확인

표준 PostgreSQL `EXPLAIN`으로 하이브리드 질의의 실행 계획을 봅니다.

In [ ]:
import re

plan = conn.execute("EXPLAIN (ANALYZE, BUFFERS) " + HYBRID_SQL, (q_emb,)).fetchall()
for (line,) in plan:
    # 질의 벡터(1024차원) 값은 '[...]'로 줄여서 보여줍니다.
    print(re.sub(r"'\[[^']*\]'", "'[...]'", line))

### 계획에서 읽을 포인트

1. **전문 검색**: `Index Scan using products_texton_idx on products p`가 보이면 texton 인덱스로 BM25 검색을 수행한 것입니다. texton의 `@@` 조건은 항상 이 인덱스 스캔으로 평가되며, `category` 조건은 같은 노드의 `Filter`로 적용됩니다.
2. **벡터 정렬**: `Sort` 노드에서 `p.emb <-> ...` 거리로 정렬하면, 앞 단계의 조건을 통과한 후보만 거리를 계산한 것입니다. 전문 검색과 그래프 조건으로 후보가 크게 줄어든 경우 플래너는 이 경로를 고릅니다.
3. **그래프**: Cypher 질의는 그래프 토폴로지 테이블(`v_product_topo`, `v_brand_topo`, `e_made_by_topo`)에 대한 조인으로 펼쳐져, 다른 조건과 함께 하나의 계획으로 최적화됩니다.

조건(`WHERE b.hot`, `category`, 전문 검색어)을 바꿔 가며 실행 계획이 어떻게 달라지는지 비교해 보세요.